# 01 · Preparación de datos

**Proyecto:** Olist, marketplace brasileño de e-commerce · **Fuente:** [Brazilian E-Commerce Public Dataset by Olist (Kaggle)](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)

Este notebook toma los 9 archivos originales y produce **`data/df_EDA.csv`**, el archivo que usan el EDA, el modelo de satisfacción y el dashboard de Power BI.

**Pasos:** perfil de calidad → limpieza por tabla → integración controlando la granularidad → variables derivadas → exportación y validación.

> Para correrlo: descargá el zip de Kaggle y descomprimilo en `data/raw/`. En Google Colab también busca la carpeta del proyecto en Drive.

```
customers ──< orders ──< order_items >── products
                │   │          └──────── sellers
                │   └──< order_payments
                └──< order_reviews
```

In [ ]:
import random
import unicodedata
from itertools import product

import numpy as np
import pandas as pd

from olist_utils import RAW, DATA, ruta, tabla

pd.set_option("display.max_columns", 40)

ARCHIVOS = {
    "orders": "olist_orders_dataset.csv",
    "items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "products": "olist_products_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "translation": "product_category_name_translation.csv",
}
raw = {nombre: pd.read_csv(ruta(archivo, RAW)) for nombre, archivo in ARCHIVOS.items()}

## 1. Perfil de calidad

Una sola tabla resume filas, duplicados y nulos de cada archivo (en lugar de imprimir `head`, `tail`, `info` y `describe` de cada uno).

In [ ]:
CLAVES = {"orders": ["order_id"], "items": ["order_id", "order_item_id"], "payments": ["order_id", "payment_sequential"],
          "reviews": ["review_id", "order_id"], "customers": ["customer_id"], "sellers": ["seller_id"],
          "products": ["product_id"], "geolocation": None, "translation": ["product_category_name"]}

perfil = pd.DataFrame({
    nombre: {
        "filas": len(d),
        "columnas": d.shape[1],
        "filas duplicadas": d.duplicated().sum(),
        "clave repetida": d.duplicated(CLAVES[nombre]).sum() if CLAVES[nombre] else np.nan,
        "columnas con nulos": (d.isna().sum() > 0).sum(),
        "% máx. de nulos": d.isna().mean().max(),
    }
    for nombre, d in raw.items()
}).T
tabla(perfil, {"filas": "{:,.0f}", "filas duplicadas": "{:,.0f}", "clave repetida": "{:,.0f}", "% máx. de nulos": "{:.1%}"})

**Lectura del perfil**

- `geolocation` tiene muchas filas duplicadas y varias coordenadas por código postal. No se usa en el resultado final (el análisis geográfico se hace por estado).
- `reviews` tiene pedidos con más de una reseña: se conserva la más reciente.
- `orders` tiene fechas nulas en los pedidos que no completaron el ciclo (ver paso 2).
- `products` tiene ~600 productos sin categoría.

## 2. Pedidos: qué entra al análisis

Se conservan los pedidos con **las cinco fechas del ciclo** (compra, aprobación, despacho, entrega y estimada),
porque todas las métricas de logística las necesitan. La tabla muestra qué se excluye y por qué.

In [ ]:
orders = raw["orders"].copy()
FECHAS_PEDIDO = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
                 "order_delivered_customer_date", "order_estimated_delivery_date"]
for col in FECHAS_PEDIDO:
    orders[col] = pd.to_datetime(orders[col])

orders["ciclo_completo"] = orders[FECHAS_PEDIDO].notna().all(axis=1)
resumen_estado = orders.groupby("order_status")["ciclo_completo"].agg(pedidos="size", con_ciclo_completo="sum")
resumen_estado["excluidos"] = resumen_estado["pedidos"] - resumen_estado["con_ciclo_completo"]
display(resumen_estado.sort_values("pedidos", ascending=False))

orders = orders[orders["ciclo_completo"]].drop(columns="ciclo_completo")
print(f"Pedidos que entran: {len(orders):,} de {len(raw['orders']):,} ({len(orders) / len(raw['orders']):.1%})")

## 3. Limpieza por tabla

### 3.1 Reseñas: una por pedido

In [ ]:
reviews = raw["reviews"].copy()
reviews["review_creation_date"] = pd.to_datetime(reviews["review_creation_date"])
print(f"Pedidos con más de una reseña: {reviews['order_id'].duplicated().sum():,}")
reviews = reviews.sort_values("review_creation_date").drop_duplicates("order_id", keep="last")[["order_id", "review_score"]]

### 3.2 Productos: categorías en español y macro-categorías

Las 73 categorías originales (en portugués) se traducen y se agrupan en 64 categorías. Luego se agrupan en 10 macro-categorías para las vistas ejecutivas.

In [ ]:
CATEGORIA_ES = {
    "utilidades_domesticas": "Artículos para el Hogar", "perfumaria": "Perfumería", "automotivo": "Automotriz",
    "pet_shop": "Tienda de Mascotas", "papelaria": "Papelería", "moveis_decoracao": "Muebles de Decoración",
    "moveis_escritorio": "Muebles de Oficina", "ferramentas_jardim": "Herramientas para Jardín",
    "informatica_acessorios": "Accesorios de Informática", "cama_mesa_banho": "Cama, Mesa y Baño",
    "brinquedos": "Juguetes", "construcao_ferramentas_ferramentas": "Herramientas de Construcción",
    "construcao_ferramentas_construcao": "Herramientas de Construcción", "telefonia": "Telefonía",
    "beleza_saude": "Belleza y Salud", "eletronicos": "Electrónica", "bebes": "Bebés",
    "cool_stuff": "Productos Innovadores", "relogios_presentes": "Relojes y Regalos", "climatizacao": "Climatización",
    "esporte_lazer": "Deportes y Ocio", "livros_interesse_geral": "Libros de Interés General",
    "eletrodomesticos": "Electrodomésticos", "eletrodomesticos_2": "Electrodomésticos",
    "alimentos_bebidas": "Alimentos y Bebidas", "alimentos": "Alimentos y Bebidas", "bebidas": "Alimentos y Bebidas",
    "malas_acessorios": "Maletas y Accesorios", "fashion_underwear_e_moda_praia": "Ropa Interior y Moda de Playa",
    "other": "Otros", "fashion_bolsas_e_acessorios": "Bolsos y Accesorios de Moda",
    "instrumentos_musicais": "Instrumentos Musicales",
    "construcao_ferramentas_iluminacao": "Herramientas de Iluminación para Construcción",
    "livros_tecnicos": "Libros Técnicos", "construcao_ferramentas_jardim": "Herramientas para Construcción y Jardinería",
    "market_place": "Marketplace", "agro_industria_e_comercio": "Agroindustria y Comercio",
    "artigos_de_festas": "Artículos para Fiestas", "casa_conforto": "Hogar y Confort", "casa_conforto_2": "Hogar y Confort",
    "cds_dvds_musicais": "CDs, DVDs y Música", "dvds_blu_ray": "CDs, DVDs y Música",
    "industria_comercio_e_negocios": "Industria, Comercio y Negocios", "consoles_games": "Consolas y Videojuegos",
    "moveis_quarto": "Muebles para Dormitorio", "construcao_ferramentas_seguranca": "Herramientas de Seguridad para Construcción",
    "telefonia_fixa": "Telefonía Fija", "moveis_cozinha_area_de_servico_jantar_e_jardim": "Muebles para el Hogar",
    "fashion_calcados": "Calzado de Moda", "casa_construcao": "Materiales de Construcción", "audio": "Audio",
    "cine_foto": "Cine y Fotografía", "moveis_sala": "Muebles para Living", "artes": "Artes y Artesanías",
    "artes_e_artesanato": "Artes y Artesanías", "fashion_roupa_masculina": "Ropa Masculina",
    "artigos_de_natal": "Artículos de Navidad", "tablets_impressao_imagem": "Tablets, Impresión e Imagen",
    "fashion_esporte": "Moda Deportiva", "portateis_cozinha_e_preparadores_de_alimentos": "Pequeños Electrodomésticos de Cocina",
    "la_cuisine": "Pequeños Electrodomésticos de Cocina", "gastronomia": "Gastronomía", "flores": "Flores",
    "pcs": "Computadoras y PCs", "pc_gamer": "Computadoras y PCs", "portateis_casa_forno_e_cafe": "Pequeños Electrodomésticos",
    "eletroportateis": "Pequeños Electrodomésticos", "moveis_colchao_e_estofado": "Sillones, Colchones y Tapicería",
    "sinalizacao_e_seguranca": "Señalización y Seguridad", "fashion_roupa_feminina": "Ropa Femenina",
    "fraldas_higiene": "Pañales e Higiene", "livros_importados": "Libros Importados", "musica": "Música",
    "fashion_roupa_infanto_juvenil": "Ropa Infantil", "seguros_e_servicos": "Seguridad y Servicios",
}
MACRO = {
    "Moda": ["Ropa Femenina", "Ropa Masculina", "Ropa Infantil", "Calzado de Moda", "Moda Deportiva",
             "Ropa Interior y Moda de Playa", "Bolsos y Accesorios de Moda", "Relojes y Regalos", "Maletas y Accesorios"],
    "Hogar y Decoración": ["Muebles para Living", "Muebles para Dormitorio", "Muebles de Decoración", "Muebles de Oficina",
                           "Muebles para el Hogar", "Artículos para el Hogar", "Hogar y Confort", "Cama, Mesa y Baño",
                           "Flores", "Climatización", "Sillones, Colchones y Tapicería"],
    "Automotriz": ["Automotriz"],
    "Construcción": ["Materiales de Construcción", "Herramientas de Construcción", "Herramientas de Seguridad para Construcción",
                     "Herramientas para Construcción y Jardinería", "Herramientas para Jardín",
                     "Herramientas de Iluminación para Construcción"],
    "Tecnología": ["Computadoras y PCs", "Accesorios de Informática", "Tablets, Impresión e Imagen", "Consolas y Videojuegos",
                   "Electrónica", "Telefonía", "Telefonía Fija", "Audio"],
    "Belleza y Cuidado Personal": ["Belleza y Salud", "Perfumería"],
    "Electrodomésticos": ["Electrodomésticos", "Pequeños Electrodomésticos", "Pequeños Electrodomésticos de Cocina"],
    "Cuidado Infantil": ["Bebés", "Pañales e Higiene"],
    "Mascotas": ["Tienda de Mascotas"],
    # el resto (libros, música, juguetes, deportes, papelería, alimentos, etc.) va a "Mercado"
}
MACRO_POR_CATEGORIA = {cat: macro for macro, cats in MACRO.items() for cat in cats}

products = raw["products"][["product_id", "product_category_name"]].copy()
products["product_category_name"] = products["product_category_name"].fillna("other").str.strip().str.lower()
sin_traduccion = set(products["product_category_name"]) - set(CATEGORIA_ES)
assert not sin_traduccion, f"Categorías sin traducir: {sin_traduccion}"
products["product_category_name_es"] = products["product_category_name"].map(CATEGORIA_ES)
products["macro_category"] = products["product_category_name_es"].map(MACRO_POR_CATEGORIA).fillna("Mercado")
print(f"{products['product_category_name'].nunique()} categorías originales → "
      f"{products['product_category_name_es'].nunique()} en español → {products['macro_category'].nunique()} macro-categorías")

### 3.3 Vendedores: seudónimos legibles

Olist publica a los vendedores anonimizados (`seller_id` es un hash de 32 caracteres), así que un ranking de vendedores no se puede leer.
Se le asigna a cada uno un **seudónimo ficticio** ("adjetivo + sustantivo"), siempre el mismo gracias a la semilla fija.

> ⚠️ Los seudónimos son **inventados**: no corresponden a empresas reales. Así se aclara también en el dashboard.

In [ ]:
ADJETIVOS = ("Amazing Brilliant Creative Dynamic Efficient Fantastic Glorious Harmonious Incredible Joyful Kind Lively "
             "Magnificent Noble Outstanding Perfect Quick Radiant Superb Thrilling Unique Vibrant Wonderful Xenial Youthful "
             "Zenith Zealous Ultimate Supreme Bold Clever Daring Energetic Fearless Genuine Helpful Imaginative Jubilant "
             "Knowledgeable Limitless Motivated Notable Optimistic Proactive Reliable Savvy Talented Unstoppable Versatile Wise "
             "Extraordinary Adventurous Ambitious Astonishing Captivating Charismatic Competent Decisive Delightful Devoted "
             "Enthusiastic Fascinating Friendly Gracious Industrious Inventive Passionate Persistent Resilient Resplendent "
             "Spectacular Strategic Tremendous Valiant Visionary Zesty").split()
SUSTANTIVOS = ("Solutions Innovations Creations Designs Developments Enterprises Ventures Inventions Technologies Industries "
               "Markets Businesses Corporations Agencies Studios Workshops Laboratories Factories Companies Groups Teams "
               "Partnerships Alliances Networks Communities Consortiums Syndicates Unions Federations Confederations Projects "
               "Ideas Platforms Firms Distributors Retailers Wholesalers Manufacturers Concepts Developers Producers Suppliers "
               "Traders Strategies Brands Systems Innovators Investments Collaborations Cooperatives Collectives Engagements "
               "Holdings Connections Dynamics Enterprise Coordinations Managers Specialists Channels Providers Creators Launches "
               "Crafts Blueprints Installations Constructors Magicians Orchestrations").split()


def seudonimos(ids, semilla=42):
    combinaciones = [f"{a} {s}" for a, s in product(ADJETIVOS, SUSTANTIVOS)]
    random.seed(semilla)
    random.shuffle(combinaciones)
    assert len(combinaciones) >= len(ids), "Faltan combinaciones de nombres"
    return dict(zip(ids, combinaciones))


sellers = raw["sellers"].copy()
sellers["seller_names"] = sellers["seller_id"].map(seudonimos(sellers["seller_id"].unique()))
assert sellers["seller_names"].is_unique
sellers[["seller_id", "seller_names", "seller_state"]].head(3)

### 3.4 Ciudades: normalización por código postal

La misma ciudad aparece escrita de varias formas ("São Paulo", "sao paulo", "sp"). Para cada código postal se toma la variante
más frecuente del nombre normalizado (minúsculas, sin tildes ni sufijos como "- sp" o "(…)").
Una sola función reemplaza el bloque que antes estaba copiado tres veces.

In [ ]:
def normalizar_texto(s: pd.Series) -> pd.Series:
    s = s.fillna("").map(lambda x: unicodedata.normalize("NFKD", x).encode("ascii", "ignore").decode())
    s = s.str.lower().str.replace(r"\s*[\(,/-].*$", "", regex=True)   # "lages - sc", "rio (rj)" → "lages", "rio"
    return s.str.replace(r"[^a-z ]", "", regex=True).str.split().str.join(" ")


def ciudad_por_cp(df: pd.DataFrame, col_cp: str, col_ciudad: str) -> pd.Series:
    """Ciudad canónica: el nombre normalizado más frecuente dentro de cada código postal."""
    norm = normalizar_texto(df[col_ciudad])
    canonica = norm.groupby(df[col_cp]).agg(lambda x: x.value_counts().index[0])
    return df[col_cp].map(canonica)


for tabla_nombre, prefijo in [("customers", "customer"), ("sellers", "seller")]:
    t = raw[tabla_nombre] if tabla_nombre == "customers" else sellers
    antes = t[f"{prefijo}_city"].nunique()
    t[f"{prefijo}_city"] = ciudad_por_cp(t, f"{prefijo}_zip_code_prefix", f"{prefijo}_city")
    print(f"{tabla_nombre}: {antes:,} variantes de ciudad → {t[f'{prefijo}_city'].nunique():,}")
customers = raw["customers"]

## 4. Integración controlando la granularidad

Cada tabla tiene su propio nivel: pedido, ítem o pago. Unirlas en una sola tabla plana produce
**una fila por combinación ítem × pago** de cada pedido. Es lo que espera Power BI, que después separa las tablas en un esquema estrella,
pero obliga a deduplicar antes de sumar (ver notebook 02). Aquí se verifica que ninguna unión multiplique filas más allá de eso.

In [ ]:
items = raw["items"].copy()
items["shipping_limit_date"] = pd.to_datetime(items["shipping_limit_date"])
payments = raw["payments"]

base = orders.merge(customers[["customer_id", "customer_unique_id", "customer_state"]], on="customer_id", how="left", validate="1:1")
plano = (base
         .merge(items, on="order_id", how="inner", validate="1:m")
         .merge(payments, on="order_id", how="inner", validate="m:m")
         .merge(reviews, on="order_id", how="left", validate="m:1")
         .merge(products, on="product_id", how="left", validate="m:1")
         .merge(sellers[["seller_id", "seller_names", "seller_state"]], on="seller_id", how="left", validate="m:1"))

esperado = (items[items["order_id"].isin(base["order_id"])].groupby("order_id").size()
            * payments.groupby("order_id").size()).dropna().sum()
assert len(plano) == esperado, "Alguna unión duplicó filas de más"
print(f"Filas: {len(plano):,} = Σ (ítems × pagos) por pedido · pedidos: {plano['order_id'].nunique():,}")
print(f"Pedidos sin ítems o sin pagos (quedan fuera): {len(base) - plano['order_id'].nunique():,}")

**Ítems con flete cero.** Hay ítems con envío gratis (`freight_value = 0`). La versión publicada del dashboard los excluye;
se mantiene esa regla para que los números coincidan, pero queda como parámetro explícito. Con `False` entran también los pedidos con envío gratis.

In [ ]:
EXCLUIR_FLETE_CERO = True  # compatibilidad con el dashboard publicado

filas_cero = (plano["freight_value"] <= 0) | (plano["price"] <= 0)
pedidos_afectados = plano.loc[filas_cero, "order_id"].nunique()
print(f"Filas con precio o flete en cero: {filas_cero.sum():,} ({pedidos_afectados:,} pedidos)")
if EXCLUIR_FLETE_CERO:
    plano = plano[~filas_cero].copy()

## 5. Variables derivadas

Se calculan aquí, una sola vez, las variables que usan el EDA y el dashboard. Cada una se calcula en su granularidad y luego se asigna a las filas.

In [ ]:
df = plano
ts = df["order_purchase_timestamp"]
DIAS = lambda a, b: (df[a] - df[b]).dt.days

# Calendario de la compra
df["day_of_month"], df["hour"], df["month"] = ts.dt.day, ts.dt.hour, ts.dt.month
df["month_year"] = ts.dt.to_period("M").astype(str)
df["day_of_week"] = ts.dt.day_name()

# Logística (días)
df["processing_time"] = DIAS("order_delivered_customer_date", "order_purchase_timestamp")   # compra → entrega
df["approval_time"] = DIAS("order_approved_at", "order_purchase_timestamp")                 # compra → aprobación
df["prep_time"] = DIAS("order_delivered_carrier_date", "order_approved_at")                 # aprobación → transportista
df["shipping_late_days"] = DIAS("order_delivered_carrier_date", "shipping_limit_date")      # > 0: el vendedor despachó tarde
df["delivery_delay"] = DIAS("order_delivered_customer_date", "order_estimated_delivery_date")  # > 0: llegó tarde
df["on_time"] = df["delivery_delay"] <= 0
df["late_delivery"] = df["delivery_delay"] > 0
df["late_delivery_label"] = df["late_delivery"].map({False: "on_time", True: "late_delivery"})

# Geografía
REGION = {**dict.fromkeys(["SP", "RJ", "MG", "ES"], "Sudeste"), **dict.fromkeys(["PR", "SC", "RS"], "Sur"),
          **dict.fromkeys(["DF", "GO", "MS", "MT"], "Centro-Oeste"),
          **dict.fromkeys(["BA", "PE", "CE", "MA", "PB", "RN", "AL", "SE", "PI"], "Nordeste"),
          **dict.fromkeys(["AM", "PA", "TO", "RO", "RR", "AP", "AC"], "Norte")}
ESTADO = {"AC": "Acre", "AL": "Alagoas", "AP": "Amapá", "AM": "Amazonas", "BA": "Bahia", "CE": "Ceará",
          "DF": "Distrito Federal", "ES": "Espírito Santo", "GO": "Goiás", "MA": "Maranhão", "MT": "Mato Grosso",
          "MS": "Mato Grosso do Sul", "MG": "Minas Gerais", "PA": "Pará", "PB": "Paraíba", "PR": "Paraná",
          "PE": "Pernambuco", "PI": "Piauí", "RJ": "Rio de Janeiro", "RN": "Rio Grande do Norte",
          "RS": "Rio Grande do Sul", "RO": "Rondônia", "RR": "Roraima", "SC": "Santa Catarina", "SP": "São Paulo",
          "SE": "Sergipe", "TO": "Tocantins"}
for lado in ("customer", "seller"):
    df[f"{lado}_region"] = df[f"{lado}_state"].map(REGION)
    df[f"{lado}_full_state"] = df[f"{lado}_state"].map(ESTADO)

In [ ]:
# Nivel pedido: cantidad de ítems y pagos, medio de pago, total pagado
por_item = df.drop_duplicates(["order_id", "order_item_id"]).groupby("order_id")
por_pago = df.drop_duplicates(["order_id", "payment_sequential"]).groupby("order_id")
pagado = por_pago["payment_value"].sum()

df["max_order_item_id"] = df["order_id"].map(por_item["order_item_id"].max())
df["max_payment_sequential"] = df["order_id"].map(por_pago["payment_sequential"].max())
df["most_frequent_payment_type"] = df["order_id"].map(por_pago["payment_type"].agg(lambda s: s.mode().iloc[0]))
df["precio_ticket"] = df["order_id"].map(pagado)
# Diferencia entre lo pagado y precio + flete: intereses de cuotas o descuentos por voucher. NO es beneficio del negocio.
df["beneficio_ticket"] = df["order_id"].map((pagado - por_item["price"].sum() - por_item["freight_value"].sum()).round(2))

RANGOS_TICKET = [(75, "Ticket < 75"), (150, "Ticket 75-150"), (350, "Ticket 150-350"), (500, "Ticket 350-500"), (np.inf, "Ticket > 500")]
df["rango_ticket"] = pd.cut(df["precio_ticket"], [-np.inf] + [c for c, _ in RANGOS_TICKET], labels=[e for _, e in RANGOS_TICKET]).astype(str)
df["VM_score"] = df["rango_ticket"].map({e: i + 1 for i, (_, e) in enumerate(RANGOS_TICKET)})

In [ ]:
# Nivel cliente: frecuencia de compra
num_pedidos = df.groupby("customer_unique_id")["order_id"].nunique()
df["num_pedidos"] = df["customer_unique_id"].map(num_pedidos)
FRECUENCIA = [(1, "Compra Única"), (3, "2-3 Compras"), (5, "4-5 Compras"), (7, "6-7 Compras"), (np.inf, "9 o más Compras")]
# nota: la etiqueta "9 o más" incluye 8 compras; se mantiene por compatibilidad (no hay clientes con exactamente 8)
df["frecuencia_compra"] = pd.cut(df["num_pedidos"], [0] + [c for c, _ in FRECUENCIA], labels=[e for _, e in FRECUENCIA]).astype(str)
df["freq_score"] = df["frecuencia_compra"].map({e: i + 1 for i, (_, e) in enumerate(FRECUENCIA)})
df["Total_score"] = df["freq_score"] + df["VM_score"]

**Scores de categoría.** El dashboard usa dos puntajes por categoría:

- `frecuencia_temporal`: días promedio entre dos compras consecutivas **de la categoría** (de cualquier cliente).
  Mide el **volumen** de la categoría, no cada cuánto recompra un cliente: las categorías grandes salen "Compra Diaria".
- `rango_precio_producto`: rango del precio del ítem.

Se mantienen por compatibilidad, con esa aclaración.

In [ ]:
df = df.sort_values(["product_category_name_es", "order_purchase_timestamp"])
df["dias_entre_compras"] = df.groupby("product_category_name_es")["order_purchase_timestamp"].diff().dt.days
media_cat = df.groupby("product_category_name_es")["dias_entre_compras"].mean().fillna(9999)
FREC_CAT = [(1, "Compra Diaria"), (7, "Compra Semanal"), (15, "Compra Quincenal"), (30, "Compra Mensual"), (np.inf, "Compra Estacional")]
frec_cat = pd.cut(media_cat, [-np.inf] + [c for c, _ in FREC_CAT], labels=[e for _, e in FREC_CAT]).astype(str)
df["frecuencia_temporal"] = df["product_category_name_es"].map(frec_cat)
df["freq_score_categoria"] = df["frecuencia_temporal"].map({e: 5 - i for i, (_, e) in enumerate(FREC_CAT)})
df["dias_entre_compras"] = df["dias_entre_compras"].fillna(9999)  # 9999 = primera compra de la categoría

RANGOS_PRECIO = [(75, "Precio < 75"), (200, "Precio 75-200"), (500, "Precio 200-500"), (1000, "Precio 500-1000"), (np.inf, "Precio > 1000")]
df["rango_precio_producto"] = pd.cut(df["price"], [-np.inf] + [c for c, _ in RANGOS_PRECIO], labels=[e for _, e in RANGOS_PRECIO], right=False).astype(str)
df["score_valor_producto"] = df["rango_precio_producto"].map({e: i + 1 for i, (_, e) in enumerate(RANGOS_PRECIO)})
df["total_score_categoria"] = df["freq_score_categoria"] + df["score_valor_producto"]

## 6. Exportación y validación

In [ ]:
COLUMNAS = [
    "order_id", "order_status", "order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
    "order_delivered_customer_date", "order_estimated_delivery_date", "customer_unique_id", "customer_state",
    "order_item_id", "product_id", "seller_id", "shipping_limit_date", "price", "freight_value", "payment_sequential",
    "payment_type", "payment_installments", "payment_value", "review_score", "product_category_name_es", "macro_category",
    "seller_names", "seller_state", "day_of_month", "hour", "month", "month_year", "day_of_week", "processing_time",
    "approval_time", "prep_time", "shipping_late_days", "delivery_delay", "on_time", "late_delivery", "customer_region",
    "seller_region", "customer_full_state", "seller_full_state", "max_order_item_id", "max_payment_sequential",
    "most_frequent_payment_type", "precio_ticket", "beneficio_ticket", "num_pedidos", "frecuencia_compra", "rango_ticket",
    "freq_score", "VM_score", "Total_score", "dias_entre_compras", "frecuencia_temporal", "freq_score_categoria",
    "rango_precio_producto", "score_valor_producto", "total_score_categoria", "late_delivery_label",
]
salida = df[COLUMNAS].sort_values(["order_purchase_timestamp", "order_id", "order_item_id", "payment_sequential"])

controles = pd.Series({
    "filas": len(salida),
    "pedidos": salida["order_id"].nunique(),
    "ítems": len(salida.drop_duplicates(["order_id", "order_item_id"])),
    "pagos": len(salida.drop_duplicates(["order_id", "payment_sequential"])),
    "clientes": salida["customer_unique_id"].nunique(),
    "facturación (R$)": salida.drop_duplicates(["order_id", "payment_sequential"])["payment_value"].sum(),
    "nulos fuera de review_score": salida.drop(columns="review_score").isna().sum().sum(),
})
assert controles["nulos fuera de review_score"] == 0
controles.map("{:,.0f}".format).to_frame("valor")

In [ ]:
destino = DATA / "df_EDA.csv"
salida.to_csv(destino, index=False)
print(f"Guardado {destino.relative_to(DATA.parent)} · {len(salida):,} filas × {salida.shape[1]} columnas")

## Resumen

| Paso | Resultado |
|---|---|
| Pedidos | Se conservan los que tienen el ciclo completo de fechas (compra → entrega) |
| Reseñas | Una por pedido (la más reciente) |
| Productos | Categorías traducidas al español y agrupadas en 10 macro-categorías |
| Vendedores | Seudónimos ficticios y reproducibles (los datos originales están anonimizados) |
| Ciudades | Nombre canónico por código postal |
| Integración | Tabla plana de ítem × pago, verificada contra el conteo esperado |
| Variables | Tiempos logísticos, retraso, región, ticket, frecuencia de compra y scores |

**Siguiente paso:** [`02_analisis_exploratorio.ipynb`](02_analisis_exploratorio.ipynb).